In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


False

In [3]:
# colab-only
!pip install --pre "giskard[openai]" pytest pytest-asyncio

Configure pytest to run async Giskard Checks tests with `pytest-asyncio`.

Two more self-contained, CI-verified examples live in
[`examples/`](https://github.com/Giskard-AI/giskard-oss/tree/main/examples) in
the `giskard-oss` repository.

## 1. Your first green test

Start with a test that needs nothing but Giskard: a function under test, a
scenario that calls it, and one deterministic check. Save this as
`test_echo.py`.

In [ ]:
# test_echo.py
import asyncio

from giskard.checks import Equals, Scenario


def echo(inputs: str) -> str:
    return inputs


async def test_echo():
    result = await (
        Scenario("echo")
        .interact(inputs="hello", outputs=echo)
        .check(Equals(name="echoes_input", target_key="trace.last.outputs", expected_value="hello"))
        .run()
    )
    result.print_report()
    assert result.passed


asyncio.run(test_echo())

## 2. Install dependencies

`pytest-asyncio` is what bridges the gap between pytest's synchronous test
runner and Giskard's async `Scenario.run()` method.

```bash
pip install --pre "giskard[openai]" pytest pytest-asyncio
```

Drop the `[openai]` extra if none of your checks call an LLM.

## 3. Configure `asyncio_mode`

Add `asyncio_mode = auto` so every `async def test_*` function runs
automatically without a per-test decorator.

**`pytest.ini`:**

```ini
[pytest]
asyncio_mode = auto
```

**`pyproject.toml`:**

```toml
[tool.pytest.ini_options]
asyncio_mode = "auto"
```

With this in place, drop the `asyncio.run(...)` line from the example above —
pytest runs the coroutine for you.

## 4. Point a test at your own system

Swap `echo` for the function that calls your application. The parameter name
matters: name it `inputs` (or `trace`) so Giskard knows what to pass. The
`assert result.passed` at the end is what turns a Giskard result into a pytest
failure — without it, pytest considers the test passed regardless of the
scenario outcome.

In [3]:
# test_chatbot.py
from giskard.checks import RegexMatching, Scenario


def my_chatbot(message: str) -> str:
    # Replace with a call into your own application.
    return "Hello! How can I help?"


async def test_greeting_response():
    scenario = (
        Scenario("greeting")
        .interact(
            inputs="Hello!",
            outputs=lambda inputs: my_chatbot(inputs),
        )
        .check(RegexMatching(pattern=r"hi|hello|hey", name="has_greeting"))
    )

    result = await scenario.run()
    assert result.passed

## 5. What a failure looks like

A red test is the output you will read most often, so it is worth seeing once.
The scenario below asserts a greeting that the agent does not produce. Running
it directly prints the report rather than raising, so you can see the check
name, its status, and the reason side by side.

Checks in a step run **sequentially** and the scenario stops at the first
non-passing one, which is why a failing test shows a single failing check and
not the whole list.

In [ ]:
import asyncio

from giskard.checks import Equals, Scenario


def unhelpful_bot(message: str) -> str:
    return "Request denied."


async def run_failing():
    return await (
        Scenario("greeting_is_friendly")
        .interact(inputs="Hello!", outputs=lambda inputs: unhelpful_bot(inputs))
        .check(
            Equals(
                name="says_hello",
                target_key="trace.last.outputs",
                expected_value="Hi there!",
            )
        )
        .run()
    )


failing = asyncio.run(run_failing())
failing.print_report()

# The same message pytest would show behind `assert result.passed`.
for step in failing.failures_and_errors:
    for message in step.format_failures():
        print(message)

## 6. Share generator config with a `conftest.py` fixture

Judged checks such as `LLMJudge` or `Groundedness` need a provider, and that
configuration belongs in one place. The `scope="session"` setting means the
generator is configured once per test run rather than once per test — which
matters as soon as your suite has dozens of LLM-backed checks.

```python
# conftest.py
import pytest
from giskard.agents.generators import Generator
from giskard.checks import set_default_generator


@pytest.fixture(scope="session", autouse=True)
def configure_generator():
    set_default_generator(Generator(model="openai/gpt-4o-mini"))
    # Or set GISKARD_CHECKS_DEFAULT_MODEL and skip this call entirely.
```

With `autouse=True` the fixture runs before any test in the session without
requiring an explicit parameter. Every test that reaches a judged check from
this point on needs `OPENAI_API_KEY` in the environment.

## 7. Parametrize for data-driven tests

Each parametrized case gets its own entry in the pytest output, so when one
question fails you can see exactly which input caused it without digging
through a combined result object.

In [5]:
import pytest
from giskard.checks import Scenario, StringMatching

test_cases = [
    ("What is the capital of France?", "Paris"),
    ("What is 2 + 2?", "4"),
    ("Who wrote Hamlet?", "Shakespeare"),
]


def my_agent(question: str) -> str:
    # Replace with a call into your own application.
    return {"What is 2 + 2?": "4"}.get(question, "Paris and Shakespeare.")


@pytest.mark.parametrize("question,expected", test_cases)
async def test_factual_answers(question, expected):
    scenario = (
        Scenario(f"factual_{question[:20]}")
        .interact(
            inputs=question,
            outputs=lambda inputs: my_agent(inputs),
        )
        .check(StringMatching(keyword=expected, name="correct_answer"))
    )

    result = await scenario.run()
    assert result.passed, f"Failed for question: {question}"

## 8. Run the tests

The `-v` flag prints each test name and its result individually, making it easy
to spot which parametrized case failed.

```bash
pytest -v
```

Expected output:

```
test_echo.py::test_echo PASSED
test_chatbot.py::test_greeting_response PASSED
test_factual.py::test_factual_answers[What is the capital of France?-Paris] PASSED
test_factual.py::test_factual_answers[What is 2 + 2?-4] PASSED
test_factual.py::test_factual_answers[Who wrote Hamlet?-Shakespeare] PASSED
```

Each parametrized case appears on its own line so failures are immediately
identifiable. Run a single file or test by name:

```bash
pytest test_chatbot.py -v
pytest -k "factual" -v
```

## Next steps

- [Async design & pytest](/oss/checks/explanation/async-and-pytest) — why
  `Scenario.run()` is async
- [CI/CD Integration](/oss/checks/how-to/ci-cd) — run the same tests in GitHub
  Actions, with a JUnit report
- [Runnable examples](https://github.com/Giskard-AI/giskard-oss/tree/main/examples) —
  two self-contained, CI-verified scripts in the `giskard-oss` repository
- [Simulate Users](/oss/checks/how-to/simulate-users) — drive multi-turn tests
  with LLM-generated inputs